In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import csv
from pathlib import Path
from collections import defaultdict

# ======== CONFIGURE YOUR ROOT FOLDER HERE ========
ROOT = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files"
OUT_SUMMARY_CSV = "config_file_groups_summary.csv"
OUT_LISTING_CSV = "config_file_groups_listing.csv"
# ================================================

TRAILING_IDX_RE = re.compile(r"^(.*?)(__\d+)(\.[^.]+)?$", flags=re.IGNORECASE)

def strip_trailing_index(base: str) -> str:
    """
    Remove trailing '__<digits>' if present (before or after the extension).
      'foo__12.yml'   -> 'foo.yml'
      'foo.yml__12'   -> 'foo.yml'
      'foo__12'       -> 'foo'
    """
    m = TRAILING_IDX_RE.match(base)
    if not m:
        return base
    stem, _, ext = m.groups()
    return stem + (ext or "")

def remove_repo_prefix(filename: str) -> str:
    """
    Flattened names often look like '<owner.repo>__<encoded>'.
    Return just the encoded part if '__' exists.
    """
    return filename.split("__", 1)[1] if "__" in filename else filename

def is_yaml(ext: str) -> bool:
    return ext in (".yml", ".yaml")

# Common script extensions (you can expand if needed)
SCRIPT_EXTS = {
    ".sh", ".bash", ".zsh", ".ksh", ".bat", ".cmd",
    ".ps1", ".psm1", ".py", ".rb", ".pl"
}

def is_build_file(encoded_lower: str) -> bool:
    """
    Treat any module build script as a Build file:
      - Literal: 'build.gradle', 'build.gradle.kts'
      - Encoded: 'gradle++build', 'build++gradle'
        (optionally followed by .gradle / .kts / .gradle.kts)
      - Edge: plain 'build' with .gradle/.kts or even no ext (rare)
    """
    base = Path(encoded_lower).name  # filename only
    ext  = Path(encoded_lower).suffix  # includes leading dot or ''

    # Literal exact names
    if base in {"build.gradle", "build.gradle.kts"}:
        return True

    # Core without final extension (to catch 'gradle++build' before extra suffixes)
    core = base[:-len(ext)] if ext else base

    # Encoded module build tokens
    if core in {"gradle++build", "build++gradle"}:
        return True

    # Sometimes the flattening yields e.g. 'gradle++build.gradle' or '.gradle.kts'
    if base.startswith("gradle++build") or base.startswith("build++gradle"):
        # e.g., 'gradle++build.gradle', 'gradle++build.gradle.kts'
        return True

    # Rare edge: 'build' with relevant extensions
    if core == "build" and ext in {"", ".gradle", ".kts", ".gradle.kts"}:
        return True

    return False

def is_other_gradle(encoded_lower: str) -> str | None:
    """
    Return bucket name for non-build Gradle scripts, or None if not Gradle.
    """
    base = Path(encoded_lower).name
    if base.endswith(".gradle.kts"):
        return "Other Gradle (*.gradle.kts)"
    if base.endswith(".gradle"):
        return "Other Gradle (*.gradle)"
    # Heuristic: tokenized gradle-like names (not specifically 'build')
    if "++gradle++" in base or base.endswith("++gradle") or base.startswith("gradle++"):
        return "Other Gradle (*.gradle)"
    return None

def classify_group(raw_relpath: str) -> str:
    """
    Classify into:
      - Build files
      - YAML/YML
      - Other Gradle (*.gradle)
      - Other Gradle (*.gradle.kts)
      - Support scripts (shell/bash & others)
      - Other files
    """
    base = Path(raw_relpath).name
    base = strip_trailing_index(base)
    encoded = remove_repo_prefix(base)
    encoded_lower = encoded.lower()
    ext = Path(encoded_lower).suffix

    if is_yaml(ext):
        return "YAML/YML"

    if is_build_file(encoded_lower):
        return "Build files (build.gradle or build.gradle.kts)"

    og = is_other_gradle(encoded_lower)
    if og:
        return og

    if ext in SCRIPT_EXTS or encoded_lower in {"gradlew", "gradlew.bat"}:
        return "Support scripts (shell/bash & others)"

    return "Other files"

def main():
    groups = defaultdict(list)
    total = 0

    for root, _, files in os.walk(ROOT):
        for fname in files:
            total += 1
            rel = os.path.relpath(os.path.join(root, fname), ROOT)
            group = classify_group(rel)
            groups[group].append(rel)

    ordered_groups = [
        "YAML/YML",
        "Build files (build.gradle or build.gradle.kts)",
        "Other Gradle (*.gradle)",
        "Other Gradle (*.gradle.kts)",
        "Support scripts (shell/bash & others)",
        "Other files",
    ]

    # Print counts
    print(f"Total files scanned (raw): {total}\n")
    for g in ordered_groups:
        print(f"{g}: {len(groups.get(g, []))}")

    # Write summary CSV
    with open(OUT_SUMMARY_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["group", "count"])
        for g in ordered_groups:
            w.writerow([g, len(groups.get(g, []))])
        w.writerow(["TOTAL (raw)", total])

    # Write listing CSV
    with open(OUT_LISTING_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["group", "relative_path"])
        for g in ordered_groups:
            for p in sorted(groups.get(g, [])):
                w.writerow([g, p])

    # --- Quick sanity checks on the samples you gave ---
    samples = [
        "thialfihar.apg__gradle++build__2",
        "felangel.flutter_and_friends__gradle++build__2.gradle",
        "konyaco.collinsdictionary__gradle++build__4.gradle",
    ]
    print("\nSample classification checks:")
    for s in samples:
        print(s, "->", classify_group(s))

    print(f"\nWrote summary to: {OUT_SUMMARY_CSV}")
    print(f"Wrote listing to: {OUT_LISTING_CSV}")

if __name__ == "__main__":
    main()


Total files scanned (raw): 78894

YAML/YML: 12667
Build files (build.gradle or build.gradle.kts): 21339
Other Gradle (*.gradle): 12722
Other Gradle (*.gradle.kts): 1768
Support scripts (shell/bash & others): 20728
Other files: 9670

Sample classification checks:
thialfihar.apg__gradle++build__2 -> Build files (build.gradle or build.gradle.kts)
felangel.flutter_and_friends__gradle++build__2.gradle -> Build files (build.gradle or build.gradle.kts)
konyaco.collinsdictionary__gradle++build__4.gradle -> Build files (build.gradle or build.gradle.kts)

Wrote summary to: config_file_groups_summary.csv
Wrote listing to: config_file_groups_listing.csv
